# HiveMind Benchmark Autopilot (Pro-Run-Zellen, Foreground + Live Google Drive Sync)

Dieses Notebook teilt die Benchmarks in **kompakte Einzel-Zellen pro Durchlauf/Slice** auf, damit Google Colab sauber durchläuft und nicht durch zu lange Gesamtlaufzeiten abbricht:
- Nach **jeder einzelnen Task × Harness-Zelle** schreibt `runner.py` atomar lokal **und** spiegelt sofort nach Google Drive (`--sync-dir`).
- Am **Ende jeder Benchmark-Zelle** wird der Gesamtstand (`core_4b_100k.json`, `polyglot_4b_100k.json`, …) nochmals verifiziert & auf Drive gesichert — danach startet die nächste Zelle.
- **Nahtloses Auto-Resume (`--resume`):** Falls Colab trotzdem trennt, einfach `1 → 2 → 3` und die letzte nicht abgeschlossene Zelle erneut ausführen: bereits erledigte `(run, task, condition)`-Zellen werden übersprungen.

### Reihenfolge:
- **3a–3c:** 4B Core Suite (`23 Tasks`) — jeweils **1 Zelle pro Run** (`Run 1/3`, `Run 2/3`, `Run 3/3`), je ~45–55 Min.
- **3d–3e:** 4B Polyglot Python (`34 Tasks`, `1 Run`) — aufgeteilt in **Teil 1/2 (Tasks 1–17)** & **Teil 2/2 (Tasks 18–34)**.
- **4a–4f (Optional):** Dasselbe Schema für `35b-mtp` (`120k Ctx`, ab 20 GB VRAM) und `DeepSWE` (nur A100 >= 35 GB).

**Fairness (`2722e92`):** Gleiches Modell, gleicher `llama-server`, identisches `_agent_timeout_s` über alle Harnesses (`300s` Core/Polyglot, `1800s` DeepSWE). `hivemind_duo` nutzt `duo_core` (18 Basis-Tools + `+Subagent` Optional = 19 Default, `send_file` request-gated).

In [ ]:
# 1) Google Drive mounten & vorhandene Checkpoints prüfen
import json
from collections import Counter
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hivemind_bench")
DRIVE_RESULTS = DRIVE_ROOT / "results"
DRIVE_LOGS = DRIVE_ROOT / "logs"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive verbunden: {DRIVE_RESULTS}")
except ImportError:
    DRIVE_RESULTS = Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
    print(f"Hinweis: google.colab nicht aktiv, lokaler Backup-Ordner: {DRIVE_RESULTS}")

existing = [p for p in sorted(DRIVE_RESULTS.glob("*.json")) if not p.name.startswith("_")]
if existing:
    print(f"\nGefundene Checkpoints in Drive ({len(existing)}):")
    for p in existing:
        try:
            d = json.loads(p.read_text(encoding="utf-8"))
            res = d.get("results", [])
            by_run = dict(sorted(Counter(int(r.get("run") or 1) for r in res).items()))
            print(f"  - {p.name:30s} | model={d.get('model')} ctx={d.get('num_ctx', '?')} rows={len(res)} runs={by_run}")
        except Exception as exc:
            print(f"  - {p.name:30s} | (nicht lesbar: {exc})")
else:
    print("\nNoch keine Checkpoints in Drive — frischer Start.")

In [ ]:
# 2) Repo aktualisieren, Drive -> Lokal vorab syncen & Basis-Setup (inkl. Polyglot & DeepSWE)
import json, shutil, subprocess, sys
from pathlib import Path

if "DRIVE_RESULTS" not in globals():
    DRIVE_ROOT = Path("/content/drive/MyDrive/hivemind_bench")
    DRIVE_RESULTS = DRIVE_ROOT / "results" if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = DRIVE_ROOT / "logs" if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)

def _run_stream(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    assert p.stdout is not None
    for line in p.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed (exit={rc}): {cmd}")

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /root | tail -1

_run_stream(["bash", "-lc", (
    "[ -d /root/HiveMind_llama/.git ] || git clone --branch bench-stable https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama; "
    "cd /root/HiveMind_llama && git fetch origin bench-stable && git checkout bench-stable && git reset --hard origin/bench-stable && "
    "ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh && "
    "chmod +x /root/rent_bootstrap.sh && git log -1 --format='[bench] commit: %h %s'"
)])

# Vorher-Sync: Alle Drive-Checkpoints in das lokale results/-Verzeichnis ziehen
local_res = Path("/root/HiveMind_llama/bench/results")
local_res.mkdir(parents=True, exist_ok=True)
def _json_rows(p):
    try:
        return len(json.loads(Path(p).read_text(encoding="utf-8")).get("results", []))
    except Exception:
        return -1
for src in sorted(DRIVE_RESULTS.glob("*.json")):
    if src.name.startswith("_"):
        continue
    dst = local_res / src.name
    src_rows = _json_rows(src)
    dst_rows = _json_rows(dst) if dst.exists() else -1
    if (not dst.exists()) or (src_rows >= 0 and src_rows >= dst_rows):
        shutil.copy2(src, dst)
        print(f"[pre-sync] {src.name} ({max(0, src_rows)} Rows) -> {dst}")

# Basis-Setup (Python 3.14, llama.cpp, Aider, Hermes, Pi + Node 22)
_run_stream(["bash", "-lc", "HM_MODEL=skip HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup"])

# Alle 3 Task-Corpora bereithalten (Core ist im Repo, Polyglot & DeepSWE werden importiert)
_poly_dir = Path("/root/HiveMind_llama/bench/tasks_polyglot")
_poly_count = len(list(_poly_dir.glob("polyglot_*/task.json")))
if _poly_count < 34:
    print(f"[setup] Polyglot nur {_poly_count}/34 Tasks vorhanden — importiere vollständigen Corpus.")
    _run_stream(["bash", "-lc", "HM_POLYGLOT_NAMES=all bash /root/rent_bootstrap.sh polyglot"])
else:
    print("[setup] Polyglot Tasks (34) bereits vollständig.")

try:
    # uv-created venvs have no pip; rent_bootstrap's deepswe() already does
    # this, keep it non-fatal so Core/Polyglot still run if the mirror is down.
    _run_stream(["bash", "-lc", "cd /root/HiveMind_llama && uv pip install --python .venv/bin/python -q hypothesis trio attrs pytest pytest-asyncio anyio"])
except RuntimeError as _dep_exc:
    print(f"[setup][warn] DeepSWE test deps not installed: {_dep_exc}")
_ds_dir = Path("/root/HiveMind_llama/bench/tasks_deepswe")
_ds_count = len(list(_ds_dir.glob("*/task.json")))
if _ds_count < 3:
    print(f"[setup] DeepSWE nur {_ds_count}/3 Tasks vorhanden — importiere Corpus.")
    _run_stream(["bash", "/root/rent_bootstrap.sh", "deepswe"])
else:
    print("[setup] DeepSWE Tasks (3) bereits vollständig.")
print(f"[setup] BEREIT: {DRIVE_RESULTS}")

In [ ]:
# 2b) OPTIONAL: Bestehende Benchmarks löschen (für komplett sauberen Neustart ohne Resume/Merge)
DELETE_EXISTING_BENCHMARKS = False  # Auf True setzen und Zelle ausführen, um alte Checkpoints/Logs/State zu löschen
import os as _reset_os, pathlib as _reset_pl, shutil as _reset_shutil, signal as _reset_sig

_reset_dirs = [
    _reset_pl.Path("/root/HiveMind_llama/bench/results"),
    _reset_pl.Path("/root/HiveMind_llama/bench/results/external"),
    _reset_pl.Path("/root/HiveMind_llama/bench/results_backup"),
    _reset_pl.Path("/root/HiveMind_llama/bench/logs_backup"),
    _reset_pl.Path("/root/HiveMind_llama/logs"),
    _reset_pl.Path("/content/drive/MyDrive/hivemind_bench/results"),
    _reset_pl.Path("/content/drive/MyDrive/hivemind_bench/results/external"),
    _reset_pl.Path("/content/drive/MyDrive/hivemind_bench/logs"),
]
_reset_files = [f for d in _reset_dirs if d.exists() for f in (*d.glob("*.json"), *d.glob("*.log"))]
for _sf in (
    _reset_pl.Path("/root/HiveMind_llama/bench_full.log"),
    _reset_pl.Path("/root/HiveMind_llama/bench_engine.log"),
    _reset_pl.Path("/root/HiveMind_llama/bench_gpu_settings.json"),
    _reset_pl.Path("/root/HiveMind_llama/settings.json"),
    _reset_pl.Path("/root/HiveMind_llama/runtime_models.json"),
    _reset_pl.Path("/root/HiveMind_llama/memory.json"),
    _reset_pl.Path("/root/HiveMind_llama/token_stats.json"),
    _reset_pl.Path("/root/HiveMind_llama/run_counter.json"),
):
    if _sf.exists():
        _reset_files.append(_sf)
_work_dir = _reset_pl.Path("/root/HiveMind_llama/bench/.work")
if not DELETE_EXISTING_BENCHMARKS:
    print(f"[reset] ÜBERSPRUNGEN ({len(_reset_files)} vorhandene Dateien behalten). Zum Löschen DELETE_EXISTING_BENCHMARKS = True setzen.")
else:
    _pid_f = _reset_pl.Path("/root/HiveMind_llama/bench_engine.pid")
    if _pid_f.exists():
        try:
            _reset_os.kill(int(_pid_f.read_text().strip()), _reset_sig.SIGTERM)
        except (OSError, ValueError):
            pass
        _pid_f.unlink(missing_ok=True)
    for _f in _reset_files:
        _f.unlink(missing_ok=True)
    if _work_dir.exists():
        _reset_shutil.rmtree(_work_dir, ignore_errors=True)
    print(f"[reset] {len(_reset_files)} Benchmark-/State-Dateien & Logs gelöscht — frischer Start ohne Alt-Daten bereit.")

In [ ]:
# 3) Helper laden (run_stage & run_autopilot) — vor 3a..4f einmal ausführen
import json as _json, os, shutil as _shutil
import subprocess as _subprocess
import sys as _sys
from pathlib import Path

if "DRIVE_RESULTS" not in globals() or "DRIVE_LOGS" not in globals():
    _dr = Path("/content/drive/MyDrive/hivemind_bench")
    DRIVE_RESULTS = (_dr / "results") if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = (_dr / "logs") if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)

CONDITIONS = "hivemind_duo,hivemind_duo_lean,aider,hermes,pi"
CTX_4B_SMALL_SUITES = 102400
CTX_35B_SMALL_SUITES = 122880
SUITES = {
    "core": "bench/tasks",
    "polyglot": "bench/tasks_polyglot",
    "deepswe": "bench/tasks_deepswe",
}


def _row_count(p):
    try:
        p = Path(p)
        if not p.is_file():
            return 0
        return len(_json.loads(p.read_text(encoding="utf-8")).get("results", []))
    except Exception:
        return 0


def _detect_vram_gb():
    try:
        out = _subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            text=True,
        ).strip().splitlines()[0]
        return float(out) / 1024.0
    except Exception:
        return 15.0


def _resolve_model(model_choice="4b", ctx_small=None, model_id=None, for_deepswe=False):
    vram_gb = _detect_vram_gb()
    use_model = str(model_choice).strip().lower()
    if use_model == "auto":
        use_model = "4b" if vram_gb < 20.0 else "35b-mtp"
    if use_model == "4b":
        resolved_model = model_id or "qwen3.5:4b-mtp"
        tag = "4b"
        ctx = 131072 if for_deepswe else int(ctx_small or CTX_4B_SMALL_SUITES)
    elif use_model == "35b-mtp":
        resolved_model = model_id or "hermes3.6:35b-a3b-uncensored-genesis-apex-mini"
        tag = "35b_mtp"
        ctx = 131072 if for_deepswe else int(ctx_small or CTX_35B_SMALL_SUITES)
    else:
        resolved_model = model_id or "qwen3.6:35b-a3b-ud"
        tag = "35b"
        ctx = 131072 if for_deepswe else int(ctx_small or CTX_35B_SMALL_SUITES)
    if for_deepswe and vram_gb >= 35.0 and vram_gb < 60.0:
        print(f"[stage] DeepSWE 128k needs a big A100; using 64k on {vram_gb:.1f} GB.")
        ctx = 65536
    return use_model, resolved_model, tag, ctx, vram_gb


def run_stage(suite, model_choice="4b", run=1, conditions=CONDITIONS,
              ctx_small=None, model_id=None, only=None, task_slice=None,
              cell_timeout=None, task_timeout=None):
    """Führt genau EINEN Durchlauf (oder Task-Slice) aus und sichert direkt auf Drive."""
    suite_key = str(suite).strip().lower()
    if suite_key not in SUITES:
        raise ValueError(f"Unbekannte Suite: {suite!r} (erlaubt: {list(SUITES)})")
    for_deepswe = (suite_key == "deepswe")
    use_model, resolved_model, tag, ctx_tokens, vram_gb = _resolve_model(
        model_choice, ctx_small=ctx_small, model_id=model_id, for_deepswe=for_deepswe
    )
    if use_model.startswith("35b") and vram_gb < 20.0:
        print(f"[stage] SKIP {suite_key} ({use_model}): {vram_gb:.1f} GB VRAM < 20 GB.")
        return
    if for_deepswe and vram_gb < 35.0:
        print(f"[stage] SKIP DeepSWE: {vram_gb:.1f} GB VRAM < 35 GB (A100 required).")
        return

    cond_list = [c.strip() for c in conditions.split(",") if c.strip()]
    if ctx_tokens < 65536 and "hermes" in cond_list:
        cond_list.remove("hermes")
        print(f"[stage] Hermes omitted at ctx={ctx_tokens}: needs >=64k.")
    stage_conditions = ",".join(cond_list)
    if not stage_conditions:
        print(f"[stage] SKIP {suite_key}: keine gueltigen Conditions.")
        return

    tasks_rel = SUITES[suite_key]
    only_str = str(only or "").strip()
    slice_label = ""
    if task_slice is not None:
        tasks_dir = Path("/root/HiveMind_llama") / tasks_rel
        all_ids = sorted(
            p.name for p in tasks_dir.iterdir()
            if p.is_dir() and (p / "task.json").is_file()
        )
        start_i, end_i = task_slice
        selected = all_ids[start_i:end_i]
        if not selected:
            print(f"[stage] SKIP {suite_key}: leerer Task-Slice {task_slice} (gesamt {len(all_ids)}).")
            return
        only_str = ",".join(selected)
        slice_label = f" | slice {start_i + 1}-{start_i + len(selected)}/{len(all_ids)}"

    stage_name = f"{suite_key}_{tag}_{ctx_tokens // 1024}k"
    out_file = Path(f"/root/HiveMind_llama/bench/results/{stage_name}.json")
    _saved = DRIVE_RESULTS / out_file.name
    out_file.parent.mkdir(parents=True, exist_ok=True)
    if _saved.is_file():
        try:
            if _saved.resolve() != out_file.resolve() and _row_count(_saved) >= _row_count(out_file):
                _shutil.copy2(_saved, out_file)
                print(f"[pre-sync] Drive -> Lokal: {_saved.name} ({_row_count(_saved)} Rows)", flush=True)
        except OSError as _pre_exc:
            print(f"[warn] Pre-Sync fehlgeschlagen: {_pre_exc}", flush=True)
    log_file = DRIVE_LOGS / f"{stage_name}.log"
    run_no = max(1, int(run))
    eff_cell_to = int(cell_timeout or (1800 if for_deepswe else 900))

    print("\n=================================================================")
    print(f"[STAGE] {stage_name} | run={run_no}{slice_label} | model={resolved_model} | ctx={ctx_tokens} | conds={stage_conditions}")
    print(f"=================================================================", flush=True)

    env = os.environ.copy()
    env.update({
        "HM_MODEL": resolved_model,
        "HM_CTX": str(ctx_tokens),
        "HM_TASKS": tasks_rel,
        "HM_RUNS": "1",
        "HM_RUN_OFFSET": str(run_no - 1),
        "HM_CONDITIONS": stage_conditions,
        "HM_CELL_TIMEOUT": str(eff_cell_to),
        "HM_TASK_TIMEOUT": str(int(task_timeout)) if task_timeout else "",
        "HM_ONLY": only_str,
        "HM_OUT": str(out_file),
        "HM_SYNC_DIR": str(DRIVE_RESULTS),
        "HM_RESUME": "1",
    })

    try:
        log_file.parent.mkdir(parents=True, exist_ok=True)
        _log_ctx = open(log_file, "a", encoding="utf-8")
    except OSError as _log_exc:
        print(f"[warn] Drive log unavailable ({_log_exc}); using local fallback", flush=True)
        log_file = Path("/root/HiveMind_llama/bench/logs_backup") / f"{stage_name}.log"
        log_file.parent.mkdir(parents=True, exist_ok=True)
        _log_ctx = open(log_file, "a", encoding="utf-8")

    with _log_ctx as lf:
        lf.write(f"\n=== STAGE {stage_name} run={run_no}{slice_label} START ===\n")
        proc = _subprocess.Popen(
            ["bash", "/root/rent_bootstrap.sh", "fg"], env=env,
            stdout=_subprocess.PIPE, stderr=_subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        )
        try:
            assert proc.stdout is not None
            for line in proc.stdout:
                _sys.stdout.write(line)
                _sys.stdout.flush()
                lf.write(line)
                lf.flush()
            rc = proc.wait()
        except KeyboardInterrupt:
            proc.terminate()
            proc.wait(timeout=10)
            raise
        finally:
            _eng_log = Path("/root/HiveMind_llama/bench_engine.log")
            if _eng_log.exists():
                try:
                    _shutil.copy2(_eng_log, DRIVE_LOGS / f"{stage_name}_engine.log")
                except OSError:
                    pass
            _saved = DRIVE_RESULTS / out_file.name
            if out_file.is_file():
                try:
                    if _saved.resolve() != out_file.resolve():
                        _shutil.copy2(out_file, _saved)
                except OSError as _sync_exc:
                    print(f"[warn] Final Drive-Sync fehlgeschlagen: {_sync_exc}", flush=True)

    _chk = _saved if _saved.exists() else (out_file if out_file.exists() else None)
    _rows = _json.loads(_chk.read_text(encoding="utf-8")).get("results", []) if _chk else []
    _run_rows = [r for r in _rows if int(r.get("run") or 1) == run_no]
    _ok = sum(1 for r in _rows if r.get("ok"))
    _run_ok = sum(1 for r in _run_rows if r.get("ok"))
    _sz = f"{_chk.stat().st_size / 1024:.1f} KB" if _chk else "0 KB"
    if rc != 0:
        raise RuntimeError(f"Stage {stage_name} run={run_no} abgebrochen (exit={rc}). Gesichert: {len(_rows)} Rows ({_sz}) -> {_saved}")
    print(f"[SAVE OK] {out_file.name} | Run {run_no}: {len(_run_rows)} Rows ({_run_ok} PASS) | Gesamt: {len(_rows)} Rows ({_ok} PASS) | Drive: {_saved} ({_sz})", flush=True)


def run_autopilot(model_choice="4b", core_runs=3, polyglot_runs=1,
                  deepswe_runs=0, conditions=CONDITIONS,
                  ctx_small=None, model_id=None):
    """Kompatibilitäts-Wrapper: führt Core (pro Run), Polyglot & DeepSWE sequentiell aus."""
    for r in range(1, int(core_runs) + 1):
        run_stage("core", model_choice=model_choice, run=r, conditions=conditions, ctx_small=ctx_small, model_id=model_id)
    for r in range(1, int(polyglot_runs) + 1):
        run_stage("polyglot", model_choice=model_choice, run=r, conditions=conditions, ctx_small=ctx_small, model_id=model_id)
    for r in range(1, int(deepswe_runs) + 1):
        run_stage("deepswe", model_choice=model_choice, run=r, conditions=conditions, ctx_small=ctx_small, model_id=model_id)

print(f"[helper] Bereit. Drive-Sync-Ziel: {DRIVE_RESULTS}")

In [ ]:
# 3a) 4B Core (23 Tasks) — Run 1/3 -> Save & Drive-Sync
run_stage("core", "4b", run=1)

In [ ]:
# 3b) 4B Core (23 Tasks) — Run 2/3 -> Save & Drive-Sync (setzt per --resume nahtlos fort)
run_stage("core", "4b", run=2)

In [ ]:
# 3c) 4B Core (23 Tasks) — Run 3/3 -> Save & Drive-Sync
run_stage("core", "4b", run=3)

In [ ]:
# 3d) 4B Polyglot (34 Tasks) — Teil 1/2 (Tasks 1–17) -> Save & Drive-Sync
run_stage("polyglot", "4b", run=1, task_slice=(0, 17))

In [ ]:
# 3e) 4B Polyglot (34 Tasks) — Teil 2/2 (Tasks 18–34) -> Save & Drive-Sync
run_stage("polyglot", "4b", run=1, task_slice=(17, 34))

In [ ]:
# 4a) OPTIONAL 35B-MTP Core (23 Tasks, 120k Ctx) — Run 1/3 -> Save & Drive-Sync
run_stage("core", "35b-mtp", run=1, ctx_small=122880)

In [ ]:
# 4b) OPTIONAL 35B-MTP Core (23 Tasks, 120k Ctx) — Run 2/3 -> Save & Drive-Sync
run_stage("core", "35b-mtp", run=2, ctx_small=122880)

In [ ]:
# 4c) OPTIONAL 35B-MTP Core (23 Tasks, 120k Ctx) — Run 3/3 -> Save & Drive-Sync
run_stage("core", "35b-mtp", run=3, ctx_small=122880)

In [ ]:
# 4d) OPTIONAL 35B-MTP Polyglot (34 Tasks, 120k Ctx) — Teil 1/2 (Tasks 1–17) -> Save & Drive-Sync
run_stage("polyglot", "35b-mtp", run=1, task_slice=(0, 17), ctx_small=122880)

In [ ]:
# 4e) OPTIONAL 35B-MTP Polyglot (34 Tasks, 120k Ctx) — Teil 2/2 (Tasks 18–34) -> Save & Drive-Sync
run_stage("polyglot", "35b-mtp", run=1, task_slice=(17, 34), ctx_small=122880)

In [ ]:
# 4f) OPTIONAL DeepSWE (nur A100 + 35B, 128k, 1 Run) -> Save & Drive-Sync
run_stage("deepswe", "35b", run=1)

In [ ]:
# 5) Leaderboard über alle in Google Drive gespeicherten Benchmarks
import json as _json_lb, subprocess as _subprocess_lb, sys as _sys_lb
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive as _colab_drive
        _colab_drive.mount("/content/drive")
    except ImportError:
        pass

if "DRIVE_RESULTS" not in globals():
    _dr_lb = Path("/content/drive/MyDrive/hivemind_bench/results")
    _loc_lb = Path("/root/HiveMind_llama/bench/results")
    _bak_lb = Path("/root/HiveMind_llama/bench/results_backup")
    if Path("/content/drive/MyDrive").exists():
        DRIVE_RESULTS = _dr_lb
    elif _loc_lb.exists() and any(not p.name.startswith("_") for p in _loc_lb.glob("*.json")):
        DRIVE_RESULTS = _loc_lb
    else:
        DRIVE_RESULTS = _bak_lb

_repo_lb = Path("/root/HiveMind_llama")
_analyze_lb = _repo_lb / "bench" / "analyze.py"
if not _analyze_lb.is_file():
    print("[leaderboard] bench/analyze.py nicht lokal vorhanden — klone bench-stable shallow...")
    if (_repo_lb / ".git").exists():
        _subprocess_lb.run(["git", "-C", str(_repo_lb), "fetch", "origin", "bench-stable"], check=False)
        _subprocess_lb.run(["git", "-C", str(_repo_lb), "checkout", "bench-stable"], check=False)
    else:
        _subprocess_lb.run(
            ["git", "clone", "--depth", "1", "--branch", "bench-stable",
             "https://github.com/BoredLuzo/HiveMind_llama.git", str(_repo_lb)],
            check=True,
        )

_venv_py = _repo_lb / ".venv" / "bin" / "python"
_py_lb = str(_venv_py) if _venv_py.is_file() else _sys_lb.executable

files = [p for p in sorted(DRIVE_RESULTS.glob("*.json")) if not p.name.startswith("_")] if DRIVE_RESULTS.exists() else []
if not files:
    print(f"Noch keine JSON-Ergebnisse in {DRIVE_RESULTS} gefunden.")
else:
    _combined_export = {"source_dir": str(DRIVE_RESULTS), "per_file": {}, "combined": None}
    _valid_files = []
    for f in files:
        print(f"\n==================== {f.name} ====================")
        try:
            _data = _json_lb.loads(f.read_text(encoding="utf-8"))
        except Exception as _read_exc:
            print(f"[warn] Überspringe unlesbare Datei {f.name}: {_read_exc}")
            continue
        _res = _data.get("results") or []
        _by_run = {}
        for _r in _res:
            _rn = int(_r.get("run") or 1)
            _by_run.setdefault(_rn, []).append(_r)
        _uniq_tasks = len({str(_r.get("task") or "") for _r in _res if _r.get("task")})
        print(f"[Overview] model={_data.get('model')} | ctx={_data.get('num_ctx', '?')} | total={len(_res)} cells | unique_tasks={_uniq_tasks}")
        _run_overview = {}
        for _rn in sorted(_by_run):
            _rlist = _by_run[_rn]
            _t_cnt = len({str(_r.get("task") or "") for _r in _rlist if _r.get("task")})
            _c_cnt = len({str(_r.get("condition") or "") for _r in _rlist if _r.get("condition")})
            _p_cnt = sum(1 for _r in _rlist if _r.get("ok"))
            print(f"  Run {_rn}: {len(_rlist)} cells ({_t_cnt} tasks x {_c_cnt} conditions) | {_p_cnt}/{len(_rlist)} PASS")
            _run_overview[str(_rn)] = {"cells": len(_rlist), "tasks": _t_cnt, "conditions": _c_cnt, "passes": _p_cnt}
        _subprocess_lb.run(
            [_py_lb, str(_analyze_lb), "--results", str(f), "--by-model"],
            check=False,
        )
        _json_proc = _subprocess_lb.run(
            [_py_lb, str(_analyze_lb), "--results", str(f), "--by-model", "--json"],
            capture_output=True, text=True, encoding="utf-8", check=False,
        )
        if _json_proc.returncode == 0 and _json_proc.stdout.strip():
            try:
                _combined_export["per_file"][f.name] = {
                    "model": _data.get("model"),
                    "num_ctx": _data.get("num_ctx"),
                    "runs": _run_overview,
                    "summary": _json_lb.loads(_json_proc.stdout),
                }
                _valid_files.append(str(f))
            except ValueError:
                pass
    if _valid_files:
        _all_proc = _subprocess_lb.run(
            [_py_lb, str(_analyze_lb), "--results", *_valid_files, "--by-model", "--json"],
            capture_output=True, text=True, encoding="utf-8", check=False,
        )
        if _all_proc.returncode == 0 and _all_proc.stdout.strip():
            try:
                _combined_export["combined"] = _json_lb.loads(_all_proc.stdout)
            except ValueError:
                pass
        _summary_out = DRIVE_RESULTS / "_leaderboard_summary.json"
        _summary_out.write_text(_json_lb.dumps(_combined_export, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        print(f"\n[SAVE OK] Zusammenfassung exportiert -> {_summary_out}")

In [ ]:
# 6) Server & VRAM sauber freigeben
!bash /root/rent_bootstrap.sh stop